In [ ]:
# ── Setup Colab — correr solo la primera vez ──────────────────────────────────
import os, sys

if not os.path.exists("/content/Curva_de_bonos"):
    !git clone https://github.com/maximogrimoldi/Curva_de_bonos.git /content/Curva_de_bonos

os.chdir("/content/Curva_de_bonos")
sys.path.insert(0, "/content/Curva_de_bonos")

!pip install -q -r requirements.txt
print("Listo — podés correr el resto del notebook.")

# Estimación de Curvas Spot y Z-Spreads — Argentina
**Fecha de valuación:** 10 de abril de 2026  
**Metodología:** Nelson-Siegel-Svensson (NSS)  
**Familias analizadas:** Bonos Soberanos USD · CER (Tasa Real) · Dollar Linked

## Marco Teórico

### ¿Qué es una curva spot?
Una tasa spot s(t) es la tasa de interés que aplica hoy para un flujo de caja que ocurre exactamente en t años. A diferencia de la TIR —que promedia todos los flujos de un bono en una sola tasa— la curva spot asigna una tasa distinta a cada plazo. Esto permite valuar correctamente cualquier flujo futuro y calcular tasas forward implícitas.

### ¿Por qué Nelson-Siegel-Svensson en lugar de Bootstrap?
El bootstrap extrae tasas spot bono por bono de forma secuencial, produciendo una curva discreta que solo existe en los plazos donde hay bonos. NS/NSS, en cambio, calibra una función continua y suave usando todos los bonos simultáneamente, minimizando el error cuadrático entre yields teóricas y de mercado. Esto lo hace más robusto ante precios con ruido y produce una curva evaluable en cualquier plazo.

### Modelos NS y NSS

**Nelson-Siegel (NS, 4 parámetros)** — usado para USD y Dollar Linked:
$$s(t) = \beta_0 + \beta_1 \cdot \varphi(t,\tau) + \beta_2 \cdot [\varphi(t,\tau) - e^{-t/\tau}]$$

**Nelson-Siegel-Svensson (NSS, 6 parámetros)** — usado para CER por su forma en "S" con dos jorobas:
$$s(t) = \beta_0 + \beta_1 \cdot \varphi(t,\tau_1) + \beta_2 \cdot [\varphi(t,\tau_1) - e^{-t/\tau_1}] + \beta_3 \cdot [\varphi(t,\tau_2) - e^{-t/\tau_2}]$$

donde $\varphi(t,\tau) = \frac{1 - e^{-t/\tau}}{t/\tau}$

Los parámetros tienen interpretación económica directa:
- **β₀**: nivel de largo plazo — tasa a la que converge la curva cuando t → ∞
- **β₁**: pendiente — diferencia entre tasa corta y larga; negativo implica curva invertida en el origen
- **β₂, β₃**: curvaturas — controlan la presencia de "jorobas" en el tramo corto y largo respectivamente
- **τ₁, τ₂**: escalas temporales — determinan en qué plazo aparece cada joroba

La función objetivo incorpora una **penalidad sobre la tasa forward**:
$$\min \sum_i w_i(y_i^{\text{obs}} - y_i^{\text{modelo}})^2 + \lambda \sum_t \max(-\Delta f(t), 0)^2$$
donde el término $\lambda$ penaliza caídas en la forward, eliminando artefactos de "pico" sin forzar monotonía estricta.

### Z-Spread
El Z-Spread de una ON corporativa es el spread constante Z que sumado a toda la curva spot soberana iguala el precio de mercado de la ON:

$$P^{ON} = \sum_t \frac{CF_t}{(1 + s(t) + Z)^t}$$

Mide el rendimiento incremental que ofrece la ON por encima del soberano — es una medida del riesgo crediticio corporativo relativo al soberano del mismo tipo de moneda.

### Las tres familias de bonos argentinos
| Familia | Moneda de pago | Qué ajusta | Tasa que se estima | Modelo |
|---------|---------------|-----------|-------------------|----|
| Hard Dollar | USD | Nada | Tasa en USD | NS |
| CER | ARS | Inflación (IPC) | Tasa real en ARS | **NSS** |
| Dollar Linked | ARS | Tipo de cambio oficial | Tasa en USD implícita | NS |

## Setup

In [ ]:
import sys
import warnings
import matplotlib
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from main import main, CURVE_REGISTRY, SETTLEMENT_DATE, _run_curve_section, print_stats_table
from src import NSSCurve, SpreadEngine, Visualizer, data_loader

matplotlib.rcParams['figure.dpi'] = 120
print(f"Proyecto cargado correctamente. Settlement: {SETTLEMENT_DATE}")

---
## 1. Curva Soberana — Globales USD

Bonos utilizados: GD29, GD30, GD35, GD38, GD41, GD46  
Precio expresado como % del valor nominal original (base 100).  
Las ONs de referencia son YPF 2029 y Telecom 2031.

In [ ]:
main(curve="USD")

### Interpretación — USD
- La curva es **normal empinada en el tramo corto y plana en el largo**: sube de 5.2% (GD29, 3.25 años) a ~9.2% a los 9 años, y luego se aplana en 10–10.5% hasta los 20 años.
- **β₀ = 14%** es la tasa de largo plazo implícita del modelo — el mercado le asigna a Argentina un spread soberano USD elevado en el muy largo plazo.
- **β₁ = −9.9%** refleja que la tasa de corto plazo (β₀+β₁ ≈ 4.1%) arranca muy por debajo del largo plazo; la curva sube rápido en el tramo 3–9 años.
- El aplanamiento entre GD38–GD41–GD46 indica que el riesgo país ya está mayormente descontado en el tramo medio; el largo queda anclado a una convergencia gradual hacia spreads EM normales.
- Los errores negativos sistemáticos en el tramo largo (GD38 −276 bps, GD46 −285 bps) reflejan una limitación del NS estándar: la curva real tiene forma de "escalón" que 4 parámetros no ajustan perfectamente cuando τ toca su bound.
- Z-Spreads positivos (YMCQOD +193 bps, TLC10D +79 bps) confirman que ambas ONs rinden por encima del soberano, como corresponde.

---
## 2. Curva Soberana — CER (Tasa Real ARS)

**Modelo:** Nelson-Siegel-Svensson (NSS, 6 parámetros)  
**Universo:** 8 bonos — X15Y6, TZX26, TX26, TZXM7 (cupón cero / ultra-corto), TX28, TX31 (BONCER fijo), DICP (amortizable 5.83%), PARP (step-up bullet 2038)  
**Precios:** Dirty Price en ARS ajustado por CER (Valor Técnico indexado). CER = 312.45 al 10/04/2026.  
**Tasas estimadas:** TIR real anual por encima de la inflación.  

Se usa NSS en lugar de NS porque la curva CER tiene **forma en "S"**: tasas negativas en el tramo ultra-corto ($< 1$ año), transición abrupta a positivo entre 0.5y y 1y, y una reaceleración en el tramo largo (DICP 6.4%). Una sola joroba (NS) no puede capturar ambos extremos sin artefactos.  
Las ONs de referencia son Telecom Serie J (TLCJO) e IRSA Propiedades (IRCFO).

In [ ]:
main(curve="CER")

### Interpretación — CER (NSS)

**Parámetros calibrados:** β₀=+7.00%, β₁=−25.00%, β₂=+15.97%, β₃=−9.81%, τ₁=0.30, τ₂=3.00

- **Tasa inicial s(0) ≈ β₀+β₁ = −18%**: la curva arranca fuertemente negativa. A t=0.10a (X15Y6), la tasa ya es −12.83%. Esto refleja una **represión financiera en el tramo ultra-corto**: los Lecers con vencimiento en mayo-junio 2026 cotizan por encima de su Valor Técnico, comprimiendo el retorno real a territorio muy negativo.

- **τ₁ = 0.30 años (≈ 4 meses):** la primera joroba (β₂=+16%) genera la rápida salida del negativo. El NSS transiciona hacia positivo antes de los 2 años, consistente con los BONCER (TX28, TX31) rindiendo 2.6–2.7%.

- **Reaceleración en el largo:** DICP (6.41% a 7.7 años) y β₀=7% anclan el largo plazo real en el rango 7–7.5%. Esto implica que el mercado exige una **tasa real de largo plazo elevada** para bonos complejos con exposición CER prolongada.

- **R² = 0.903, RMSE = 200 bps:** ajuste aceptable dada la estructura inusual de los datos. Los errores grandes en TX26 (+366 bps) y TZXM7 (−237 bps) no son fallas del modelo sino fricciones de mercado:
  - TX26 rinde 3.48% a 0.58a porque tiene un cupón inminente (May-26) que distorsiona el YTM vs. la tasa spot pura.
  - DICP (+253 bps): prima de complejidad estructural (amortización + indexación); no representa la tasa spot "limpia" del tramo 7–8 años.

- **Z-Spreads CER:** TLCJO +1423 bps, IRCFO +113 bps. El spread amplísimo de TLCJO refleja la **iliquidez estructural del mercado CER corporativo**: hay pocos compradores de deuda real ARS a largo plazo fuera del sector financiero regulado.

---
## 3. Curva Soberana — Dollar Linked

Bonos utilizados: BPY26 (BOPREAL), TTJ26 y TTD26 (Duales), TZV27, TZV28, TV30 (Soberanos DL)  
Las yields de BOPREALs se ajustan restando **200 bps** para limpiar el riesgo BCRA y usarlos como proxy soberano DL.  
Las tasas estimadas son **tasas en USD implícitas** — rendimiento en términos del tipo de cambio oficial.  
Las ONs de referencia son VSCIO y MGC30.

In [ ]:
main(curve="DL")

### Interpretación — Dollar Linked
- La curva es **muy empinada y decreciente**: arranca en ~9% (BOPREALs y Duales, tramo 0–0.6 años) y converge a **~0% en el largo plazo** (β₀ = 0.10%).
- **TV30 al 0.81%** implica que el mercado prácticamente no exige prima sobre el crawling peg oficial en ese horizonte: el mercado anticipa tipo de cambio estable.
- **β₁ = +9.52%** genera la pendiente: la tasa de cortísimo plazo (β₀+β₁ ≈ 9.6%) refleja la prima de riesgo cambiario y de cepo en el tramo muy corto.
- **RMSE alto (183 bps) esperado**: el universo DL es heterogéneo (soberanos + duales con opcionalidad + BOPREALs con riesgo BCRA). Los errores opuestos de TZV27 (−277 bps) y TZV28 (+257 bps) reflejan diferencias de liquidez entre bonos del mismo emisor, no fallas del modelo.
- **VSCIO con Z-Spread negativo (−107 bps)**: la ON cotiza por debajo del soberano DL, posiblemente por garantía explícita o demanda institucional concentrada — una distorsión de mercado que amerita análisis adicional.
- **MGC30 con +477 bps**: riesgo crediticio o iliquidez muy elevados sobre el soberano DL.

---
## Conclusiones

### Comparación entre familias

| Familia | Modelo | Pendiente | β₀ (largo plazo) | RMSE | R² | ONs Z-Spread |
|---------|--------|-----------|-----------------|------|----|-------------|
| Hard Dollar | NS | Normal empinada → flat | 14.00% | $1.46 | 0.955 | +79 a +193 bps |
| CER (real) | **NSS** | "S" invertida (−13% → +7%) | 7.00% | 200 bps | 0.903 | +113 a +1423 bps |
| Dollar Linked | NS | Muy empinada → ~0% | 0.10% | 183 bps | 0.706 | −107 a +477 bps |

### Hallazgos principales
1. **CER — Represión financiera en el ultra-corto:** X15Y6 y TZX26 con TIRs de −12.83% y −9.27% reflejan que los Lecers cotizan por encima de su Valor Técnico. El mercado acepta yields negativas para asegurarse liquidez CER antes de mediados de 2026. La curva NSS captura esta dinámica con β₁=−25% y τ₁=0.30a.

2. **CER — El costo de capital de largo plazo es elevado:** β₀=7% y DICP al 6.41% indican que el mercado exige tasas reales altas para exposición CER de largo plazo — coherente con un contexto de incertidumbre sobre la trayectoria futura de la inflación.

3. **USD — Riesgo país concentrado en el tramo medio:** La curva sube de 5.2% (GD29) a ~9.2% a 9 años y luego se aplana. β₀=14% implica que el mercado le asigna a Argentina un spread soberano USD elevado en el muy largo plazo, pero ya lo descuenta progresivamente.

4. **DL — Convergencia a cero:** TV30 al 0.81% y β₀=0.10% indican que el mercado prácticamente no exige prima sobre el crawling peg en el horizonte 4-5 años. La curva empinada en el corto refleja el riesgo de cepo y no el riesgo cambiario de largo plazo.

5. **Selección de modelos:** NS funciona bien cuando la curva es monotona o tiene una sola joroba (USD, DL). NSS es necesario para CER porque la represión financiera en el ultra-corto genera una forma en "S" que requiere dos jorobas independientes para ser parametrizada sin artefactos.

6. **VSCIO con Z-Spread negativo (−107 bps):** la ON cotiza por debajo del soberano DL, posiblemente por garantía explícita o demanda institucional concentrada — una distorsión de mercado que amerita análisis adicional.